# 🏥 CareFlow — Notebook 13: Patient Flow Funnel & Dropout Analysis
## Clinical Conversion Funnels, Step Attrition & Left Without Being Seen (LWBS)

**Author:** CareFlow Analytics Team  
**Dataset:** `careflow_analytics/output/fct_patient_journeys.parquet` and `stg_ehr_events.parquet`

---

### Objectives:
1. Map the 7-stage ED care funnel from Initial Arrival to Final Disposition.
2. Calculate step-by-step conversion and dropout rates.
3. Quantify Left Without Being Seen (LWBS) and Left Against Medical Advice (AMA) risk factors.
4. Model 6-month longitudinal trends in inpatient admissions and transfers.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')

stg_df = pd.read_parquet('../careflow_analytics/output/stg_ehr_events.parquet')
fct_df = pd.read_parquet('../careflow_analytics/output/fct_patient_journeys.parquet')

print(f"Loaded {len(fct_df):,} patient journeys.")
fct_df.head()


In [ ]:
# 1. Funnel Stages Sizing
stages = [
    'Registration',
    'Triage',
    'Doctor Consultation',
    'X-Ray',
    'Treatment'
]

funnel_counts = []
total_cases = len(fct_df)

for stg in stages:
    cnt = stg_df[stg_df['activity_name'] == stg]['case_id'].nunique()
    pct = (cnt / total_cases) * 100
    funnel_counts.append({'Stage': stg, 'Patients': cnt, 'Retention %': pct})

funnel_df = pd.DataFrame(funnel_counts)
print("Patient Flow Funnel Conversion Table:")
funnel_df


In [ ]:
# 2. Visualize Patient Flow Funnel
plt.figure(figsize=(10, 5))
bars = plt.barh(funnel_df['Stage'], funnel_df['Patients'], color='#06b6d4', alpha=0.85)

for bar in bars:
    w = bar.get_width()
    plt.text(w + 10, bar.get_y() + bar.get_height()/2, f"{int(w)} ({w/total_cases*100:.1f}%)",
             va='center', color='#fff', fontweight='bold')

plt.gca().invert_yaxis()
plt.title("7-Stage Clinical Patient Flow Retention Funnel")
plt.xlabel("Patient Encounter Volume")
plt.xlim(0, total_cases * 1.15)
plt.tight_layout()
plt.show()
